<img width="8%" alt="Archivist" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/Biblioth%C3%A8que_Inguimbertine_salle_X.jpg/1920px-Biblioth%C3%A8que_Inguimbertine_salle_X.jpg" style="border-radius: 15%">

# Archivist - Extract
<a href="https://github.com/mbasri/archivist">Give Feedback</a> | <a href="https://github.com/mbasri/archivist">Bug report</a>

**Tags:** #archivist #extract #markitdown #nas #text #python

**Author:** [Mohamed BASRI](https://github.com/mbasri)

**Last update:** 2026-04-06 (Created: 2026-04-06)

**Description:** Scans the NAS directory, converts each file to plain text using MarkItDown (100% local), and saves the result into the `extracted/` folder with a normalized Linux-friendly filename. Already extracted files are skipped (hash-based).

**References:**
- [MarkItDown](https://github.com/microsoft/markitdown)
- [Archivist project](https://github.com/mbasri/archivist)

## Setup

### Install dependencies

In [ ]:
import sys
import subprocess
from pathlib import Path

project_root = Path().resolve()
project_root = project_root.parent if project_root.name == "notebooks" else project_root

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-r", str(project_root / "requirements.txt")],
    capture_output=True, text=True
)
print(result.stderr[-2000:] if result.returncode != 0 else "✓ Dependencies installed")

## Input

### Import libraries

In [ ]:
import hashlib
import json
import os
import re
import unicodedata
import datetime
from collections import defaultdict
from dotenv import load_dotenv
from markitdown import MarkItDown

load_dotenv(project_root / ".env")
print("✓ .env loaded")

### Setup Variables
- `nas_path`: directory to scan
- `supported_extensions`: file types to extract
- `extracted_dir`: where converted text files are saved

In [ ]:
nas_path  = os.getenv("NAS_PATH")

supported_extensions = [
    ".pdf", ".docx", ".pptx", ".ppt", ".pptm",
    ".xlsx", ".xls", ".csv",
    ".txt", ".md",
    ".html", ".htm",
    ".epub", ".xml", ".json",
]

extracted_dir  = project_root / os.getenv("EXTRACTED_DIR", "extracted")
extract_index  = project_root / os.getenv("EXTRACT_INDEX", "extract_index.json")

# Target chunk size used to estimate number of chunks at indexing time
target_chunk_size = int(os.getenv("CHUNK_SIZE", 512))

extracted_dir.mkdir(exist_ok=True)

print(f"NAS path          : {nas_path}")
print(f"Extracted dir     : {extracted_dir}")
print(f"Extract index     : {extract_index}")
print(f"Target chunk size : {target_chunk_size} tokens")

## Model

### Load extract index

In [ ]:
if extract_index.exists():
    with open(extract_index, "r") as f:
        index = json.load(f)
else:
    index = {}

print(f"Index loaded: {len(index)} file(s) already extracted")

### Scan and filter new files

In [ ]:
def hash_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_filename(name: str) -> str:
    # Remove accents: é → e, à → a, ç → c ...
    name = unicodedata.normalize("NFD", name)
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")
    # Lowercase
    name = name.lower()
    # Replace anything not alphanumeric, dot or dash with underscore
    name = re.sub(r"[^a-z0-9._-]", "_", name)
    # Collapse multiple underscores
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

all_files = defaultdict(int)
to_extract = []

for file in Path(nas_path).rglob("*"):
    if not file.is_file():
        continue
    ext = file.suffix.lower()
    all_files[ext] += 1
    if ext not in supported_extensions:
        continue
    h = hash_file(file)
    if h not in index:
        to_extract.append((file, h))

print(f"Total files scanned  : {sum(all_files.values())}")
print(f"Already extracted    : {len(index)}")
print(f"To extract           : {len(to_extract)}")

### Extract and save
Convert each file to Markdown text with MarkItDown and save it to `extracted/` with a normalized filename.

In [ ]:
md        = MarkItDown()
extracted = 0
errors    = []
total     = len(to_extract)

for i, (file, h) in enumerate(to_extract, 1):
    try:
        text = md.convert(str(file)).text_content

        if not text.strip():
            print(f"  [{i}/{total}] [SKIP] {file.name} → empty content")
            continue

        stem     = normalize_filename(file.stem)
        out_name = f"{stem}.md"
        out_path = extracted_dir / out_name
        if out_path.exists():
            out_path = extracted_dir / f"{stem}_{h[:8]}.md"

        out_path.write_text(text, encoding="utf-8")

        estimated_chunks = max(1, len(text) // (target_chunk_size * 4))

        stat = file.stat()
        index[h] = {
            "source":               str(file),
            "extracted":            str(out_path),
            "extension":            file.suffix.lower(),
            "source_size_bytes":    stat.st_size,
            "extracted_size_bytes": out_path.stat().st_size,
            "char_count":           len(text),
            "word_count":           len(text.split()),
            "target_chunk_size":    target_chunk_size,
            "estimated_chunks":     estimated_chunks,
            "extracted_at":         datetime.datetime.now().isoformat(timespec="seconds"),
        }
        with open(extract_index, "w") as f:
            json.dump(index, f, indent=2)

        print(f"  [{i}/{total}] [OK]  {file.name} → {out_path.name}  ({len(text):,} chars, ~{estimated_chunks} chunks)")
        extracted += 1

    except Exception as e:
        print(f"  [{i}/{total}] [ERR] {file.name} → {e}")
        errors.append((file, str(e)))

## Output

### Display result

In [ ]:
print(f"Files extracted : {extracted}")
print(f"Errors          : {len(errors)}")
print(f"Output folder   : {extracted_dir}")

if errors:
    print("\nFailed files:")
    for file, err in errors:
        print(f"  - {file.name}: {err}")